# Sensoriamento Inteligente — Acessibilidade às UBS em Campina Grande (PB)

**Disciplina: Sensoriamento Inteligente e Dados Geoespaciais - 2026.2**

**Professores: Francisco Dantas Nobre Neto e Tiago Brasileiro Araújo**

**Alunos: Pedro Ivo Santos Leite, Lucas Matheus Santos da Silva**

**Notebook de projeto final | Dados vetoriais reais | Python / Google Colab**


**Como usar.** Escolha “Executar tudo”. Os dados de Campina Grande já estão recortados no repositório. O notebook carrega os setores do Censo 2022 e o cadastro de UBS consultado em 19/11/2026, calcula os indicadores, compara os agrupamentos e gera os mapas.

## 1. Preparar o ambiente

In [ ]:
%pip -q install "geopandas>=1.0" pyogrio folium scikit-learn matplotlib scipy requests
from pathlib import Path
import io, zipfile, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
from folium.features import GeoJsonTooltip
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.spatial.distance import cdist

PASTA = Path('resultados_ubs')
PASTA.mkdir(exist_ok=True)
MUNICIPIO, EPSG_METRICO, RAIO_M = '2504009', 31985, 2000
K_CANDIDATOS = [2, 3, 4]
FONTE_UBS = 'CNES · consulta de 19/11/2026'
BASE = "https://raw.githubusercontent.com/L42Matheus/colab-ubs/1fbaedae2155509bd7ac07bbcf21351e2a573e3a/dados/"

def carregar(nome):
    resposta = requests.get(BASE + nome, timeout=90)
    resposta.raise_for_status()
    return io.BytesIO(resposta.content)

## 2. Carregar setores e população de Campina Grande
O recorte do IBGE já reúne os polígonos e a população pelo código do setor. A projeção em metros permite calcular áreas e distâncias.

In [ ]:
setores = gpd.read_file(carregar('setores.geojson')).sort_values('CD_SETOR').reset_index(drop=True).to_crs(EPSG_METRICO)
setores['area_km2'] = setores.area / 1e6
setores['densidade_hab_km2'] = setores.populacao / setores.area_km2
print(len(setores), 'setores e', setores.populacao.sum(), 'moradores')

## 3. Localizar as UBS
Usamos o recorte municipal do CNES e conferimos quais coordenadas estão dentro da malha.

In [ ]:
cadastro = pd.read_csv(carregar('ubs.csv'), dtype={'CNES': str}).dropna(subset=['LONGITUDE','LATITUDE'])
ubs = gpd.GeoDataFrame(cadastro.rename(columns={'NOME':'nome'}), geometry=gpd.points_from_xy(cadastro.LONGITUDE, cadastro.LATITUDE), crs=4326).to_crs(EPSG_METRICO)
ubs = ubs[ubs.intersects(setores.geometry.union_all())].drop_duplicates('CNES').reset_index(drop=True)
display(ubs[['CNES','nome','BAIRRO','LATITUDE','LONGITUDE']])
print(len(ubs), 'UBS utilizadas')

## 5. Indicadores de distância, cobertura experimental e índice de prioridade

Cada setor é representado por um ponto interno (`representative_point`). A distância até a UBS mais próxima é euclidiana, no CRS métrico. O índice **IPS = 0,6 × distância normalizada + 0,4 × população normalizada** é um critério didático e **não é política oficial de saúde**.

In [ ]:
# Geometrias em EPSG métrico e pontos representativos internos.
pontos = setores.geometry.representative_point()
xy_setores = np.column_stack((pontos.x.to_numpy(), pontos.y.to_numpy()))
xy_ubs = np.column_stack((ubs.geometry.x.to_numpy(), ubs.geometry.y.to_numpy()))
from scipy.spatial import cKDTree
arvore = cKDTree(xy_ubs)
distancias, ids = arvore.query(xy_setores, k=1)
setores['distancia_ubs_m'] = distancias
setores['ubs_mais_proxima'] = ubs.iloc[ids]['nome'].to_numpy()
setores['ubs_2km'] = [len(v) for v in arvore.query_ball_point(xy_setores, r=RAIO_M)]

def minmax_serie(s):
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi):
        return pd.Series(np.nan,index=s.index)
    if hi == lo:
        return pd.Series(0.0,index=s.index)
    return (s-lo)/(hi-lo)

setores['IPS'] = (0.6 * minmax_serie(setores['distancia_ubs_m']) +
                  0.4 * minmax_serie(setores['populacao'].clip(lower=0)))
setores['distancia_maior_2km'] = setores['distancia_ubs_m'] > RAIO_M
print('Distância mediana (m):', round(setores['distancia_ubs_m'].median(),1))
print('Setores > 2 km:', int(setores['distancia_maior_2km'].sum()))
validos_pop = setores.dropna(subset=['populacao'])
if validos_pop['populacao'].sum() > 0:
    percentual = 100 * validos_pop.loc[validos_pop.distancia_maior_2km,'populacao'].sum() / validos_pop.populacao.sum()
    print(f'População em setores cujo ponto representativo está >2 km: {percentual:.1f}% (dos setores com população conhecida)')
else:
    print('Sem população válida; percentual populacional não calculável.')
display(setores[['CD_SETOR','populacao','distancia_ubs_m','ubs_2km','IPS']].head())

## 6. Aprendizado não supervisionado: K-Means

A IA agrupa setores usando distância à UBS, densidade populacional e número de UBS no raio de 2 km. **Clusters não possuem ordem natural**: a classificação em prioridade menor/intermediária/maior será feita *após* o ajuste, segundo a média do IPS de cada cluster. O IPS não entra na matriz de treinamento.

In [ ]:
atributos = ['distancia_ubs_m','densidade_hab_km2','ubs_2km']
dados = setores.loc[setores[atributos + ['populacao','IPS']].notna().all(axis=1)].copy()
dados = dados.loc[(dados['area_km2'] > 0) & (dados['populacao'] >= 0)].copy()
if len(dados) < 8: raise ValueError('São necessários ao menos 8 setores com dados completos.')
scaler = StandardScaler()
X = scaler.fit_transform(dados[atributos])
resultados = []
for k in K_CANDIDATOS:
    if len(dados) <= k: continue
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    rotulos = km.fit_predict(X)
    if len(set(rotulos)) < 2: continue
    resultados.append({'k':k, 'silhouette':silhouette_score(X,rotulos),
                       'davies_bouldin':davies_bouldin_score(X,rotulos),
                       'inercia':km.inertia_})
aval = pd.DataFrame(resultados)
if aval.empty: raise ValueError('Não foi possível obter clusters distintos; revise atributos.')
display(aval.round(3))
melhor_k = int(aval.sort_values(['silhouette','davies_bouldin'], ascending=[False,True]).iloc[0]['k'])
modelo = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
dados['cluster'] = modelo.fit_predict(X)
medias_ips = dados.groupby('cluster')['IPS'].mean().sort_values()
nomes = ['Prioridade relativa menor', 'Prioridade relativa intermediária', 'Prioridade relativa maior'] if melhor_k == 3 else [f'Faixa relativa {i+1}/{melhor_k}' for i in range(melhor_k)]
rotular = dict(zip(medias_ips.index,nomes))
dados['classe_relativa'] = dados['cluster'].map(rotular)
print('K selecionado por Silhouette:',melhor_k)
display(dados.groupby(['cluster','classe_relativa'])[atributos + ['IPS','populacao']].mean().round(2))

## 7. Métricas das UBS encontradas

In [ ]:
# 1. Calcular os índices Silhoutte e  Davies-Bouldin

fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].plot(aval['k'],aval['silhouette'],marker='o')
axes[0].set(title='Silhouette por número de clusters',xlabel='K',ylabel='Silhouette')
axes[0].grid(alpha=.2)
axes[1].plot(aval['k'],aval['davies_bouldin'],marker='o')
axes[1].set(title='Davies-Bouldin por número de clusters',xlabel='K',ylabel='Davies-Bouldin (menor é melhor)')
axes[1].grid(alpha=.2)
plt.tight_layout(); plt.savefig(PASTA/'metricas_clusters.png',dpi=160);plt.show()

resumo = dados.groupby('classe_relativa').agg(
    setores=('CD_SETOR','count'),
    populacao=('populacao','sum'),
    distancia_media_m=('distancia_ubs_m','mean'),
    distancia_mediana_m=('distancia_ubs_m','median'),
    IPS_medio=('IPS','mean')).reset_index()
display(resumo.round(2))
resumo.to_csv(PASTA/'resumo_clusters.csv', index=False)
aval.to_csv(PASTA/'metricas_kmeans.csv',index=False)

## 8. Mapas temáticos (arquivos PNG)

In [ ]:
fig, ax = plt.subplots(figsize=(11,9))
setores.plot(column='distancia_ubs_m',cmap='YlOrRd',legend=True,linewidth=0.05,edgecolor='gray',ax=ax)
ubs.plot(ax=ax,color='navy',markersize=13,marker='+',label='UBS mapeada')
ax.set_title('Distância euclidiana à UBS mais próxima — Campina Grande/PB')
ax.set_axis_off();ax.legend();plt.tight_layout()
plt.savefig(PASTA/'mapa_distancia.png',dpi=170,bbox_inches='tight');plt.show()

fig, ax = plt.subplots(figsize=(11,9))
dados.plot(column='cluster',categorical=True,cmap='Set2',legend=True,linewidth=0.05,edgecolor='gray',ax=ax)
ubs.plot(ax=ax,color='black',markersize=11,marker='+',label='UBS mapeada')
ax.set_title(f'Agrupamentos K-Means (K={melhor_k}) — apenas setores com dados completos')
ax.set_axis_off();ax.legend();plt.tight_layout()
plt.savefig(PASTA/'mapa_clusters.png',dpi=170,bbox_inches='tight');plt.show()

## 9. Mapas interativos HTML (Folium), Endereços e Distâncias

In [ ]:
mapa_dados = dados[['CD_SETOR','populacao','distancia_ubs_m','ubs_2km','IPS','classe_relativa','geometry']].copy().to_crs(4326)
oeste, sul, leste, norte = mapa_dados.total_bounds
mapa = folium.Map(tiles=None)
mapa.fit_bounds([[sul, oeste], [norte, leste]])
mapa_dados['distancia_ubs_m'] = mapa_dados['distancia_ubs_m'].round(0)
mapa_dados['IPS'] = mapa_dados['IPS'].round(3)
mapa_dados['populacao'] = mapa_dados['populacao'].astype(int)
cores = ['#2a9d8f','#f4a261','#e76f51','#8d5db5']
ordem = medias_ips.index.tolist()
cor_por_cluster = dict(zip(ordem,cores[:len(ordem)]))
cor_por_classe = {rotular[k]:cor_por_cluster[k] for k in ordem}
folium.GeoJson(mapa_dados.to_json(),name='Setores / K-Means',
    style_function=lambda feat:{'fillColor':cor_por_classe[feat['properties']['classe_relativa']],
                                'color':'#555','weight':0.3,'fillOpacity':0.65},
    tooltip=GeoJsonTooltip(fields=['CD_SETOR','classe_relativa','populacao','distancia_ubs_m','ubs_2km','IPS'],
      aliases=['Setor','Grupo','População','Distância (m)','UBS 2 km','IPS'])).add_to(mapa)
for _, u in ubs.to_crs(4326).iterrows():
    folium.CircleMarker([u.geometry.y,u.geometry.x],radius=4,color='blue',fill=True,
                        tooltip=str(u['nome'])).add_to(mapa)
folium.LayerControl().add_to(mapa)
mapa.save(str(PASTA/'mapa_interativo.html'))
display(mapa)

### Conexões entre setores e UBS
A camada de linhas mostra uma amostra de metade dos setores. Cada linha liga o ponto usado no cálculo à unidade mais próxima.

In [ ]:
origens = pontos.to_crs(4326)
destinos = ubs.to_crs(4326).geometry
linhas = folium.FeatureGroup(name='Conexões (amostra)', show=False)
for i in range(0, len(origens), 2):
    a, b = origens.iloc[i], destinos.iloc[ids[i]]
    folium.PolyLine([(a.y,a.x),(b.y,b.x)], weight=1, tooltip=f'{distancias[i]:.0f} m').add_to(linhas)
linhas.add_to(mapa)
mapa.save(str(PASTA/'mapa_interativo.html'))
display(mapa)

### Google My Maps · ruas e satélite
Crie um mapa no [Google My Maps](https://www.google.com/maps/d/), importe setores_google.csv e ubs_google.csv em camadas separadas. Os arquivos ficam em resultados_ubs/ e no ZIP de entrega. Escolha WKT para os setores e LATITUDE/LONGITUDE para as UBS. Para colorir os setores, agrupe o estilo por classe_relativa. Para exibir o mapa nesta célula, cole o link com mid= no campo abaixo e execute. Para compartilhar a visualização, ajuste o acesso do mapa no My Maps.

In [ ]:
camada = mapa_dados.drop(columns='geometry').copy()
camada['WKT'] = mapa_dados.geometry.to_wkt(rounding_precision=6)
camada.to_csv(PASTA/'setores_google.csv', index=False)
ubs[['CNES','nome','BAIRRO','LATITUDE','LONGITUDE']].to_csv(PASTA/'ubs_google.csv', index=False)

In [ ]:
from IPython.display import IFrame
from urllib.parse import urlparse, parse_qs, urlencode

link_google = '' # @param {type:"string"}
if link_google.strip():
    ids_mapa = parse_qs(urlparse(link_google.strip()).query).get('mid')
    if ids_mapa:
        display(IFrame('https://www.google.com/maps/d/embed?' + urlencode({'mid': ids_mapa[0]}), width='100%', height=600))
    else:
        print('Cole o link do My Maps que contém mid= na URL.')

### Distâncias entre UBS (km)

In [ ]:
matriz = pd.DataFrame(cdist(xy_ubs, xy_ubs)/1000, index=ubs.CNES, columns=ubs.CNES)
matriz.to_csv(PASTA/'distancias_entre_ubs_km.csv')
display(matriz.iloc[:5,:5].round(2))

### Simular até três novos pontos em um raio de 2 km
Cada rodada escolhe o setor que alcança mais moradores ainda além do raio. Os locais são hipóteses para avaliação em campo.

In [ ]:
xy = xy_setores
raio = RAIO_M
fora = distancias > raio
populacao = setores.populacao.to_numpy(dtype=int)
vizinhos = cdist(xy, xy) <= raio
pendentes = fora.copy()
locais = pontos.to_crs(4326)
linhas = [[0, 'Rede cadastrada', None, None, 0, populacao[pendentes].sum()]]
for rodada in range(1, 4):
    ganhos = vizinhos @ (populacao * pendentes)
    candidato = ganhos.argmax()
    if ganhos[candidato] == 0:
        break
    pendentes &= ~vizinhos[candidato]
    local = locais.iloc[candidato]
    linhas.append([rodada, setores.iloc[candidato].CD_SETOR, local.y, local.x,
                   ganhos[candidato], populacao[pendentes].sum()])

cenarios = pd.DataFrame(linhas, columns=['Novos pontos', 'Setor', 'Latitude', 'Longitude',
                              'Moradores alcançados na rodada', 'Moradores além de 2 km'])
cenarios.to_csv(PASTA/'cenarios.csv', index=False)
display(cenarios)

## 10. Exportação de dados e geração do pacote de entrega

In [ ]:
# 1. Para evitar conflito case-insensitive ao gravar no GPKG, removemos a coluna 'AREA_KM2' original do IBGE

for df_temp in [setores, dados]:
    if 'AREA_KM2' in df_temp.columns and 'area_km2' in df_temp.columns:
        df_temp.drop(columns=['AREA_KM2'], inplace=True)

# Ajustando tipos de dados para evitar conflito com o driver GPKG/pyogrio
for df_temp in [setores, dados]:
    if 'area_km2' in df_temp.columns:
        df_temp['area_km2'] = df_temp['area_km2'].astype(float)
    if 'populacao' in df_temp.columns:
        df_temp['populacao'] = pd.to_numeric(df_temp['populacao'], errors='coerce')
    if 'densidade_hab_km2' in df_temp.columns:
        df_temp['densidade_hab_km2'] = df_temp['densidade_hab_km2'].astype(float)
    if 'distancia_ubs_m' in df_temp.columns:
        df_temp['distancia_ubs_m'] = df_temp['distancia_ubs_m'].astype(float)
    if 'IPS' in df_temp.columns:
        df_temp['IPS'] = df_temp['IPS'].astype(float)

# Todas as áreas originais com indicadores, inclusive setores sem população,
# junto apenas setores usados na IA e respectivos clusters.

setores.to_file(PASTA/'setores_indicadores.gpkg',layer='setores',driver='GPKG')
dados.to_file(PASTA/'setores_clusters.gpkg',layer='clusters',driver='GPKG')
ubs.to_file(PASTA/'ubs_mapeadas.gpkg',layer='ubs',driver='GPKG')
dados.drop(columns='geometry').sort_values('IPS',ascending=False).to_csv(
    PASTA/'setores_prioritarios.csv', index=False,encoding='utf-8-sig')

nota = f"""Projeto: Acessibilidade geográfica às UBS — Campina Grande/PB
IBGE Censo: 2022; código municipal: {MUNICIPIO}
Data de processamento: {pd.Timestamp.now().isoformat()}
Fonte de pontos UBS: {FONTE_UBS}
Número de pontos válidos: {len(ubs)}
Setores na malha: {len(setores)}
Setores com atributos completos para K-Means: {len(dados)}
CRS de análise: EPSG:{EPSG_METRICO}
Raio experimental: {RAIO_M} metros
K selecionado: {melhor_k}
Limitações: distância em linha reta; ponto representativo por setor;
coordenadas cadastrais sujeitas a imprecisão; prioridade relativa não é indicador oficial.
"""
(PASTA/'LEIA_ME.txt').write_text(nota,encoding='utf-8')
zip_final = Path('entrega_sensoriamento_ubs.zip')
with zipfile.ZipFile(zip_final,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for arq in PASTA.iterdir():
        if arq.suffix.lower() in {'.gpkg','.csv','.png','.html','.txt'}:
            z.write(arq,arcname=arq.name)
print(nota)
print('Pacote criado:',zip_final)
try:
    from google.colab import files
    files.download(str(zip_final))
except ImportError:
    print('Fora do Colab: arquivo salvo em',zip_final)

## 11. Discussão e interpretação para o relatório

1. Os setores mais distantes coincidem com os mais populosos? Compare mapas e médias dos grupos.
2. O número de UBS a 2 km diferencia os agrupamentos? Compare médias e dispersão.
3. O valor de **K** selecionado gera grupos espacialmente interpretáveis? Lembre que K-Means não impõe continuidade geográfica.
4. Qual é a fração de setores sem população informada? Como isso pode afetar os resultados?
5. Como mudariam os indicadores se o raio experimental fosse **1 km ou 3 km**?
6. Em uma extensão futura, como ruas, relevo, transporte público e capacidade de atendimento alterariam a análise?

**Referências:** IBGE, Censo Demográfico 2022 / Agregados por Setores Censitários e Malha de Setores; Ministério da Saúde, CNES; OpenStreetMap Contributors, banco de dados geográfico; Pedregosa et al., *Scikit-learn: Machine Learning in Python* (2011).

**Declaração metodológica:** dados censitários referem-se a 2022 e as coordenadas das UBS provêm do CNES consultado em 19/11/2026. A diferença temporal e a precisão das coordenadas limitam a interpretação. Não conclua que um setor está desassistido apenas por estar longe de uma unidade mapeada.